## Creating the halo catalogue

Transfer files from the `agora` Globus data store. I had to transfer theses two at a time. For each of the 197 halo lightcone slice files, load them in read-only mode, save the entries with masses above the mass threshhold of $M_{500c}\ge3\times10^{14}$, compute some basic statistics about the slice and discard the rest of the file. Save these entries in a separate file with appropriate name and the statistics to a simple JSON file.

In [32]:
import numpy as np
import os

root_directory = "/home/apb86/"
data_directory = os.path.join(root_directory, "rds/hpc-work")
output_directory = "/home/apb86/rds/hpc-work/halo_catalogue"

os.makedirs(output_directory, exist_ok=True)

# --- Extract and save per-slice catalogues ---

for slice_num in range(100, 163):
    filename = f"haloslc/haloslc_rot_{slice_num}_v050223.npz"
    working_file_path = os.path.join(data_directory, filename)

    with np.load(working_file_path, allow_pickle=True, mmap_mode="r") as x:
        mask = x["totm500"] >= 1e13
        filtered = {key: x[key][mask] for key in x.files}

    output_path = os.path.join(output_directory, f"halos_rot_{slice_num:03d}_m500gt1e13.npz")
    np.savez(output_path, **filtered)
    print(f"Slice {slice_num:3d}: {mask.sum()} halos saved -> {output_path}")

# --- Concatenate into a single catalogue ---

all_data = {}

for slice_num in range(100, 163):
    input_path = os.path.join(output_directory, f"halos_rot_{slice_num:03d}_m500gt1e13.npz")
    with np.load(input_path, allow_pickle=True) as x:
        if not all_data:
            all_data = {key: [x[key]] for key in x.files}
        else:
            for key in x.files:
                all_data[key].append(x[key])

combined = {key: np.concatenate(arrays, axis=0) for key, arrays in all_data.items()}

catalogue_path = os.path.join(output_directory, "halo_catalogue_rot_4-81_m500gt1e13.npz")
np.savez(catalogue_path, **combined)

total = len(combined["totm500"])
print(f"\nCombined catalogue: {total} halos -> {catalogue_path}")

Slice 100: 193658 halos saved -> /home/apb86/rds/hpc-work/halo_catalogue/halos_rot_100_m500gt1e13.npz
Slice 101: 196325 halos saved -> /home/apb86/rds/hpc-work/halo_catalogue/halos_rot_101_m500gt1e13.npz
Slice 102: 185809 halos saved -> /home/apb86/rds/hpc-work/halo_catalogue/halos_rot_102_m500gt1e13.npz
Slice 103: 189351 halos saved -> /home/apb86/rds/hpc-work/halo_catalogue/halos_rot_103_m500gt1e13.npz
Slice 104: 192809 halos saved -> /home/apb86/rds/hpc-work/halo_catalogue/halos_rot_104_m500gt1e13.npz
Slice 105: 182619 halos saved -> /home/apb86/rds/hpc-work/halo_catalogue/halos_rot_105_m500gt1e13.npz
Slice 106: 187000 halos saved -> /home/apb86/rds/hpc-work/halo_catalogue/halos_rot_106_m500gt1e13.npz
Slice 107: 189939 halos saved -> /home/apb86/rds/hpc-work/halo_catalogue/halos_rot_107_m500gt1e13.npz
Slice 108: 178390 halos saved -> /home/apb86/rds/hpc-work/halo_catalogue/halos_rot_108_m500gt1e13.npz
Slice 109: 183889 halos saved -> /home/apb86/rds/hpc-work/halo_catalogue/halos_rot

BadZipFile: File is not a zip file

# Precprocess the data

## Point source masking

The CIB map is to have sources exceeding $2\;\text{mJy}$ masked with single pixel masks.

In [ ]:
def get_mdpl2_halo_cat(get_velocities = True):
    halo_cat_fname = 'haloc/v122921/halos_extracted_with_velocities_refined_minM500c1e+13_maxM500c1e+20_minz0.0_maxz4.0.npz.npy'


    halo_cat = np.load(halo_cat_fname, allow_pickle=True)
    mdpl2_ra, mdpl2_dec, mdpl2_z, mdpl2_m200c, mdpl2_m500c, mdpl2_vlos, mdpl2_vtht, mdpl2_vphi = halo_cat.T


    if get_velocities:
        return mdpl2_ra, mdpl2_dec, mdpl2_z, mdpl2_m200c, mdpl2_m500c, mdpl2_vlos, mdpl2_vtht, mdpl2_vphi
    else:
        return mdpl2_ra, mdpl2_dec, mdpl2_z, mdpl2_m200c, mdpl2_m500c




def get_cluster_mask_radius(m500c):


    """
    please change this as you like -- based on experiment beam, etc. --. Generally should be fine for a 1 arcmin experiment.
    """
    
    if m500c<1e14:
        cluster_mask_rad = 3.
    elif m500c>=1e14 and m500c<3e14:
        cluster_mask_rad = 5.
    elif m500c>=3e14 and m500c<5e14:
        cluster_mask_rad = 8.
    else:
        cluster_mask_rad = 10.


    return cluster_mask_rad


def get_point_source_mask_in_healpix(freq, hmap_Mjy_per_sr, threshold_mjy_freq0, threshold2_mjy_freq0 = None, freq0 = 150., spec_index = 3.4, full_sky = True, ang_res_am = None, return_flux_map_in_mjy = False):


    if full_sky:
        nside = H.get_nside(hmap_Mjy_per_sr)
        pix_area = H.nside2resol(nside)**2.
    else:
        assert ang_res_am is not None
        ang_res_rad = np.radians(ang_res_am/60.)
        pix_area = ang_res_rad**2.


    hmap_Mjy = np.copy( hmap_Mjy_per_sr ) * pix_area
    MJy_mJy = 1e9
    hmap_mjy = hmap_Mjy * MJy_mJy
    ###imshow(hmap_mjy, vmax = threshold_mjy_freq0); colorbar(); show(); sys.exit()


    scaling = (freq / freq0) ** spec_index
    threshold_mjy = threshold_mjy_freq0 * scaling
    ###print(threshold_mjy, threshold_mjy_freq0); sys.exit()


    if threshold2_mjy_freq0 is None:
        mask_pixels = np.where(hmap_mjy >= threshold_mjy)
    else:
        threshold2 = threshold2_mjy_freq0 * scaling
        mask_pixels = np.where( (hmap_mjy >= threshold_mjy) & (hmap_mjy<threshold2) )


    if full_sky:
        mask_pixels = mask_pixels[0]


    if return_flux_map_in_mjy:
        return mask_pixels, hmap_mjy
    else:
        return mask_pixels




def get_apodised_mdpl2_cluster_mask(nside, m500c_threshold = 2e14, cluster_lmz_dic = None, howmanythetaforclusters = -1, apodise = True, store_mask = True, expname = None): #tsz mask


    import copy
    print('\n\tget apodised cluster mask.')


    #change this Omar. Use your Agora ra/dec/mass/redshifts. You do not need the vecloity entries.
    mdpl2_ra, mdpl2_dec, mdpl2_z, mdpl2_m200c, mdpl2_m500c, mdpl2_vlos, mdpl2_vtht, mdpl2_vphi = get_mdpl2_halo_cat()


    if m500c_threshold != -1:
        clus_inds = np.where(mdpl2_m500c>=m500c_threshold)[0]
    else: #get cluster mask based on redshift
        assert expname is not None
        print('\tget cluster mask based on redshift')
        redshifts = cluster_lmz_dic['redshift']
        dz = np.diff(redshifts)[0]
        lim_M500c = cluster_lmz_dic['M500c'] * 1e14
        clus_inds = []
        if (1): #z = 0 to z = 0.1
            inds = np.where( (mdpl2_z<redshifts[0]) )[0]
            passed_inds = np.where(mdpl2_m500c[inds]>lim_M500c[0])[0]
            detected_mass_inds = inds[passed_inds]
            clus_inds.extend( detected_mass_inds )
        for zcntr, zzz in enumerate( redshifts ):
            inds = np.where( (mdpl2_z>=zzz) & (mdpl2_z<zzz+dz))[0]
            curr_lim_M500c = lim_M500c[zcntr]
            passed_inds = np.where(mdpl2_m500c[inds]>curr_lim_M500c)[0]
            detected_mass_inds = inds[passed_inds]


            if len(detected_mass_inds)>0:
                actual_det_M500_in_cat = min( mdpl2_m500c[detected_mass_inds] ) / 1e14
            else:                
                actual_det_M500_in_cat = -1


            print('\t\t\tz = %s; total masked = %s; Min masses: Limit = %s; Actual = %s' %(zzz, len(detected_mass_inds), curr_lim_M500c/1e14, actual_det_M500_in_cat))
            clus_inds.extend( detected_mass_inds )


    ###print( len(clus_inds) ); sys.exit()
    print('\t\ttotal clusters to be masked = %s' %(len(clus_inds))); ##sys.exit()


    if just_return_masked_inds:
        clus_inds = np.asarray(clus_inds)
        return clus_inds, mdpl2_ra[clus_inds], mdpl2_dec[clus_inds], mdpl2_m500c[clus_inds]


    h, omega_m = 0.6774, 0.3089
    cosmo = FlatLambdaCDM(H0 = h*100., Om0 = omega_m)


    if howmanythetaforclusters != -1: #get theta500c now
        cluster_mask_radius_am_arr = []        
        for cntr, iii in enumerate( clus_inds ):
            if cntr%1000==0: print(cntr)
            tmpc500c = concentration.concentration(mdpl2_m500c[iii], '500c', mdpl2_z[iii])
            m500cval, r500cval, c500cval = mass_defs.changeMassDefinition(mdpl2_m500c[iii], tmpc500c, mdpl2_z[iii], '500c', '500c', profile='nfw')
            r500cval_mpc = r500cval/1e3
            
            ang_dia_dist = cosmo.comoving_distance(mdpl2_z[iii])/(1.+mdpl2_z[iii])


            #from IPython import embed; embed(); sys.exit()
            theta500cval_am = np.degrees( r500cval_mpc/ang_dia_dist.value ) * 60.


            cluster_mask_radius_am = int( theta500cval_am * howmanythetaforclusters )+1
            cluster_mask_radius_am_arr.append( cluster_mask_radius_am )
            ##print(len(cluster_mask_radius_am_arr))


        if (1): #refined cluster_mask_radius_am_arr to few set of radii
            cluster_mask_radius_am_arr = np.asarray(cluster_mask_radius_am_arr)
            cluster_mask_radius_am_arr_mod = np.zeros_like(cluster_mask_radius_am_arr)
            cluster_mask_radius_am_arr_mod[cluster_mask_radius_am_arr<=5.] = 5.
            cluster_mask_radius_am_arr_mod[(cluster_mask_radius_am_arr>5.) & (cluster_mask_radius_am_arr<=10.)] = 8.
            cluster_mask_radius_am_arr_mod[(cluster_mask_radius_am_arr>10.) & (cluster_mask_radius_am_arr<=20.)] = 15.
            cluster_mask_radius_am_arr_mod[(cluster_mask_radius_am_arr>20.) & (cluster_mask_radius_am_arr<=50.)] = 35.
            cluster_mask_radius_am_arr_mod[(cluster_mask_radius_am_arr>50.) & (cluster_mask_radius_am_arr<=100.)] = 75.
            cluster_mask_radius_am_arr_mod[cluster_mask_radius_am_arr>100] = 100.
            cluster_mask_radius_am_arr = np.copy(cluster_mask_radius_am_arr_mod)


    #create different sets based on cluster masking radius
    print('\n\tcreate different sets based on cluster masking radius')
    npix = H.nside2npix(nside)
    hmask_dic = {}    
    for cntr, iii in enumerate( clus_inds ):
        if cntr%5000 ==0:print(cntr)
        ppp = H.ang2pix(nside, np.radians(90.-mdpl2_dec[iii]), np.radians(mdpl2_ra[iii]))
        if howmanythetaforclusters != -1:
            cluster_mask_radius_am = cluster_mask_radius_am_arr[cntr]
        else:
            cluster_mask_radius_am = get_cluster_mask_radius(mdpl2_m500c[iii])


        ivec = H.pix2vec(nside, ppp)
        disc = H.query_disc(nside, ivec, np.deg2rad(cluster_mask_radius_am/60.))
        if cluster_mask_radius_am not in hmask_dic:
            hmask_dic[cluster_mask_radius_am] = np.ones(npix)
        hmask_dic[cluster_mask_radius_am][disc] = 0.


    print(hmask_dic.keys())
    print('\t\tbinary masks obtained')
    #np.save('tsz_binary_mask.npy', hmask_dic)
    print('\n\t\tdone'); ###sys.exit()


    if apodise: #now apodise
        print('\n\tnow apodise')
        hmask_smoothed_dic = {}
        for cluster_mask_radius_am in sorted( hmask_dic ):
            print('\n\t\tmask radius = %s' %(cluster_mask_radius_am))


            if cluster_mask_radius_am<=10.:
                apod_angle_am = 10. ##cluster_mask_radius_am * 20. #10.
            else:
                apod_angle_am = 20. ##cluster_mask_radius_am * 20. #10.
            apod_angle = np.radians(apod_angle_am/60.)


            dist_smooth_angle_am = cluster_mask_radius_am ##* 3. #5.
            apod_start_dist_am = 0. ##cluster_mask_radius_am ##0.0


            dist_smooth_angle = np.radians(dist_smooth_angle_am/60.)
            apod_start_dist = np.radians(apod_start_dist_am/60.)
            apod_end_dist = apod_start_dist + apod_angle


            curr_mask = hmask_dic[cluster_mask_radius_am]
            curr_mask_smoothed = apodize_binary_mask_prof(curr_mask, dist_smooth_angle, apod_start_dist, apod_end_dist)
            curr_mask_smoothed = curr_mask_smoothed / np.max(curr_mask_smoothed)
            hmask_smoothed_dic[cluster_mask_radius_am] = curr_mask_smoothed
            #np.save('tsz_apodised_mask.npy', hmask_smoothed_dic)
        print('\n\t\tdone')
    else:
        hmask_smoothed_dic = copy.deepcopy(hmask_dic)


    print('\n\tcreate final mask')
    final_hmask_arr = list( hmask_smoothed_dic.values() )
    final_hmask = np.product( final_hmask_arr , axis = 0)
    final_hmask = final_hmask / np.max(final_hmask)


    return final_hmask




#apod mask
def apodize_binary_mask_prof(
    binary_mask, dist_smooth_angle, apod_start_dist, apod_end_dist
):
    """
    Apodize a binary mask by applying a smooth profile near the boundaries.


    The profile is [x-sin(x)]/2pi in the range [0, 2pi], which is an integral
    of a cross section of the cosine kernel used in the method above, i.e.,
    apodize_binary_mask_conv. (The cross section itself is [1 + cos(x)]/2 in
    the range [-pi, pi].)


    In this method, first, a distance map is created based on the binary mask.
    In this distance map, the value of a pixel represents the distance between
    the corresponding pixel of the binary mask and the nearest masked pixel
    (value 0.0) of the mask. Then, this distance map is smoothed by a Gaussian
    kernel. Finally, the aforementioned profile is applied to the region of the
    binary mask whose counterpart in the smoothed distance map has distance
    values within a certain range.


    If the binary mask represents a field, then the method
    apodize_binary_mask_conv seems to be better than this one because the
    former seems to create a smoother mask in the sense that there is less
    power in the spectrum of the mask beyond ell of a few thousands.


    On the other hand, if the binary mask represents point sources,
    then this latter method seems to create a better-looking apodization mask
    in the sense that the smooth region around each disk masking a source looks
    more circularly symmetric for a high Nside.


    Parameters:
    -----------
    binary_mask : array
        An array that corresponds to a full-sky HEALPix map representing a
        binary mask.
    dist_smooth_angle : float
        The FWHM of the Gaussian kernel used to smooth the distance map.
        G3Units need to be used when this parameter is specified.
    apod_start_dist : float
        The region of the binary mask whose counterpart in the smoothed
        distance map has distance values smaller than the value specified by
        this parameter will not get the profile applied. G3Units need to be
        used when this parameter is specified.
    apod_end_dist : float
        The region of the binary mask whose counterpart in the smoothed
        distance map has distance values larger than the value specified by
        this parameter will not get the profile applied. G3Units need to be
        used when this parameter is specified.


    Returns:
    --------
    smooth_mask : array:
        An array that corresponds to a full-sky HEALPix map representing the
        apodized mask.
    """
    import healpy as hp


    '''
    dist_smooth_angle /= core.G3Units.rad
    apod_start_dist /= core.G3Units.rad
    apod_end_dist /= core.G3Units.rad
    '''
    net_dist = apod_end_dist - apod_start_dist


    # Construct the distance map and smooth it
    dist_map = hp.dist2holes(binary_mask)
    binary_mask[dist_map <= apod_start_dist] = 0.0


    smooth_region = (dist_map > apod_start_dist) & (dist_map < apod_end_dist)
    #20220510 - too slow for 3*nside-1
    #dist_map = hp.smoothing(dist_map, fwhm=dist_smooth_angle)
    nside = H.get_nside(binary_mask)
    dist_map = hp.smoothing(dist_map, fwhm=dist_smooth_angle, lmax = nside)


    # Apply the profile and take care of a few small things
    smooth_mask = np.array(binary_mask)
    x = (dist_map[smooth_region] - apod_start_dist) / net_dist * 2.0 * np.pi
    del dist_map
    smooth_mask[smooth_region] = (x - np.sin(x)) / (2.0 * np.pi)


    smooth_mask *= binary_mask
    smooth_mask[smooth_mask < 0.0] = 0.0
    smooth_mask[smooth_mask > 1.0] = 1.0


    del binary_mask
    return smooth_mask


import numpy as np, sys, os, healpy as H, glob


########################################
########################################
########################################
#### Cluster mask


if (1): #mdpl2 - M500c vs z mask for SPT 100d megadeep


    m500c_threshold = -1
    expname = 'SPT-3G'
    cluster_lmz_dic_fname = 'cluster_limiting_masses_M500c_vs_z.npy'
    cluster_lmz_dic = pickle.load(open(cluster_lmz_dic_fname, 'rb'))[expname]


    nside = 8192def get_mdpl2_halo_cat(get_velocities = True):
    halo_cat_fname = 'haloc/v122921/halos_extracted_with_velocities_refined_minM500c1e+13_maxM500c1e+20_minz0.0_maxz4.0.npz.npy'


    halo_cat = np.load(halo_cat_fname, allow_pickle=True)
    mdpl2_ra, mdpl2_dec, mdpl2_z, mdpl2_m200c, mdpl2_m500c, mdpl2_vlos, mdpl2_vtht, mdpl2_vphi = halo_cat.T


    if get_velocities:
        return mdpl2_ra, mdpl2_dec, mdpl2_z, mdpl2_m200c, mdpl2_m500c, mdpl2_vlos, mdpl2_vtht, mdpl2_vphi
    else:
        return mdpl2_ra, mdpl2_dec, mdpl2_z, mdpl2_m200c, mdpl2_m500c




def get_cluster_mask_radius(m500c):


    """
    please change this as you like -- based on experiment beam, etc. --. Generally should be fine for a 1 arcmin experiment.
    """
    
    if m500c<1e14:
        cluster_mask_rad = 3.
    elif m500c>=1e14 and m500c<3e14:
        cluster_mask_rad = 5.
    elif m500c>=3e14 and m500c<5e14:
        cluster_mask_rad = 8.
    else:
        cluster_mask_rad = 10.


    return cluster_mask_rad


def get_point_source_mask_in_healpix(freq, hmap_Mjy_per_sr, threshold_mjy_freq0, threshold2_mjy_freq0 = None, freq0 = 150., spec_index = 3.4, full_sky = True, ang_res_am = None, return_flux_map_in_mjy = False):


    if full_sky:
        nside = H.get_nside(hmap_Mjy_per_sr)
        pix_area = H.nside2resol(nside)**2.
    else:
        assert ang_res_am is not None
        ang_res_rad = np.radians(ang_res_am/60.)
        pix_area = ang_res_rad**2.


    hmap_Mjy = np.copy( hmap_Mjy_per_sr ) * pix_area
    MJy_mJy = 1e9
    hmap_mjy = hmap_Mjy * MJy_mJy
    ###imshow(hmap_mjy, vmax = threshold_mjy_freq0); colorbar(); show(); sys.exit()


    scaling = (freq / freq0) ** spec_index
    threshold_mjy = threshold_mjy_freq0 * scaling
    ###print(threshold_mjy, threshold_mjy_freq0); sys.exit()


    if threshold2_mjy_freq0 is None:
        mask_pixels = np.where(hmap_mjy >= threshold_mjy)
    else:
        threshold2 = threshold2_mjy_freq0 * scaling
        mask_pixels = np.where( (hmap_mjy >= threshold_mjy) & (hmap_mjy<threshold2) )


    if full_sky:
        mask_pixels = mask_pixels[0]


    if return_flux_map_in_mjy:
        return mask_pixels, hmap_mjy
    else:
        return mask_pixels




def get_apodised_mdpl2_cluster_mask(nside, m500c_threshold = 2e14, cluster_lmz_dic = None, howmanythetaforclusters = -1, apodise = True, store_mask = True, expname = None): #tsz mask


    import copy
    print('\n\tget apodised cluster mask.')


    #change this Omar. Use your Agora ra/dec/mass/redshifts. You do not need the vecloity entries.
    mdpl2_ra, mdpl2_dec, mdpl2_z, mdpl2_m200c, mdpl2_m500c, mdpl2_vlos, mdpl2_vtht, mdpl2_vphi = get_mdpl2_halo_cat()


    if m500c_threshold != -1:
        clus_inds = np.where(mdpl2_m500c>=m500c_threshold)[0]
    else: #get cluster mask based on redshift
        assert expname is not None
        print('\tget cluster mask based on redshift')
        redshifts = cluster_lmz_dic['redshift']
        dz = np.diff(redshifts)[0]
        lim_M500c = cluster_lmz_dic['M500c'] * 1e14
        clus_inds = []
        if (1): #z = 0 to z = 0.1
            inds = np.where( (mdpl2_z<redshifts[0]) )[0]
            passed_inds = np.where(mdpl2_m500c[inds]>lim_M500c[0])[0]
            detected_mass_inds = inds[passed_inds]
            clus_inds.extend( detected_mass_inds )
        for zcntr, zzz in enumerate( redshifts ):
            inds = np.where( (mdpl2_z>=zzz) & (mdpl2_z<zzz+dz))[0]
            curr_lim_M500c = lim_M500c[zcntr]
            passed_inds = np.where(mdpl2_m500c[inds]>curr_lim_M500c)[0]
            detected_mass_inds = inds[passed_inds]


            if len(detected_mass_inds)>0:
                actual_det_M500_in_cat = min( mdpl2_m500c[detected_mass_inds] ) / 1e14
            else:                
                actual_det_M500_in_cat = -1


            print('\t\t\tz = %s; total masked = %s; Min masses: Limit = %s; Actual = %s' %(zzz, len(detected_mass_inds), curr_lim_M500c/1e14, actual_det_M500_in_cat))
            clus_inds.extend( detected_mass_inds )


    ###print( len(clus_inds) ); sys.exit()
    print('\t\ttotal clusters to be masked = %s' %(len(clus_inds))); ##sys.exit()


    if just_return_masked_inds:
        clus_inds = np.asarray(clus_inds)
        return clus_inds, mdpl2_ra[clus_inds], mdpl2_dec[clus_inds], mdpl2_m500c[clus_inds]


    h, omega_m = 0.6774, 0.3089
    cosmo = FlatLambdaCDM(H0 = h*100., Om0 = omega_m)


    if howmanythetaforclusters != -1: #get theta500c now
        cluster_mask_radius_am_arr = []        
        for cntr, iii in enumerate( clus_inds ):
            if cntr%1000==0: print(cntr)
            tmpc500c = concentration.concentration(mdpl2_m500c[iii], '500c', mdpl2_z[iii])
            m500cval, r500cval, c500cval = mass_defs.changeMassDefinition(mdpl2_m500c[iii], tmpc500c, mdpl2_z[iii], '500c', '500c', profile='nfw')
            r500cval_mpc = r500cval/1e3
            
            ang_dia_dist = cosmo.comoving_distance(mdpl2_z[iii])/(1.+mdpl2_z[iii])


            #from IPython import embed; embed(); sys.exit()
            theta500cval_am = np.degrees( r500cval_mpc/ang_dia_dist.value ) * 60.


            cluster_mask_radius_am = int( theta500cval_am * howmanythetaforclusters )+1
            cluster_mask_radius_am_arr.append( cluster_mask_radius_am )
            ##print(len(cluster_mask_radius_am_arr))


        if (1): #refined cluster_mask_radius_am_arr to few set of radii
            cluster_mask_radius_am_arr = np.asarray(cluster_mask_radius_am_arr)
            cluster_mask_radius_am_arr_mod = np.zeros_like(cluster_mask_radius_am_arr)
            cluster_mask_radius_am_arr_mod[cluster_mask_radius_am_arr<=5.] = 5.
            cluster_mask_radius_am_arr_mod[(cluster_mask_radius_am_arr>5.) & (cluster_mask_radius_am_arr<=10.)] = 8.
            cluster_mask_radius_am_arr_mod[(cluster_mask_radius_am_arr>10.) & (cluster_mask_radius_am_arr<=20.)] = 15.
            cluster_mask_radius_am_arr_mod[(cluster_mask_radius_am_arr>20.) & (cluster_mask_radius_am_arr<=50.)] = 35.
            cluster_mask_radius_am_arr_mod[(cluster_mask_radius_am_arr>50.) & (cluster_mask_radius_am_arr<=100.)] = 75.
            cluster_mask_radius_am_arr_mod[cluster_mask_radius_am_arr>100] = 100.
            cluster_mask_radius_am_arr = np.copy(cluster_mask_radius_am_arr_mod)


    #create different sets based on cluster masking radius
    print('\n\tcreate different sets based on cluster masking radius')
    npix = H.nside2npix(nside)
    hmask_dic = {}    
    for cntr, iii in enumerate( clus_inds ):
        if cntr%5000 ==0:print(cntr)
        ppp = H.ang2pix(nside, np.radians(90.-mdpl2_dec[iii]), np.radians(mdpl2_ra[iii]))
        if howmanythetaforclusters != -1:
            cluster_mask_radius_am = cluster_mask_radius_am_arr[cntr]
        else:
            cluster_mask_radius_am = get_cluster_mask_radius(mdpl2_m500c[iii])


        ivec = H.pix2vec(nside, ppp)
        disc = H.query_disc(nside, ivec, np.deg2rad(cluster_mask_radius_am/60.))
        if cluster_mask_radius_am not in hmask_dic:
            hmask_dic[cluster_mask_radius_am] = np.ones(npix)
        hmask_dic[cluster_mask_radius_am][disc] = 0.


    print(hmask_dic.keys())
    print('\t\tbinary masks obtained')
    #np.save('tsz_binary_mask.npy', hmask_dic)
    print('\n\t\tdone'); ###sys.exit()


    if apodise: #now apodise
        print('\n\tnow apodise')
        hmask_smoothed_dic = {}
        for cluster_mask_radius_am in sorted( hmask_dic ):
            print('\n\t\tmask radius = %s' %(cluster_mask_radius_am))


            if cluster_mask_radius_am<=10.:
                apod_angle_am = 10. ##cluster_mask_radius_am * 20. #10.
            else:
                apod_angle_am = 20. ##cluster_mask_radius_am * 20. #10.
            apod_angle = np.radians(apod_angle_am/60.)


            dist_smooth_angle_am = cluster_mask_radius_am ##* 3. #5.
            apod_start_dist_am = 0. ##cluster_mask_radius_am ##0.0


            dist_smooth_angle = np.radians(dist_smooth_angle_am/60.)
            apod_start_dist = np.radians(apod_start_dist_am/60.)
            apod_end_dist = apod_start_dist + apod_angle


            curr_mask = hmask_dic[cluster_mask_radius_am]
            curr_mask_smoothed = apodize_binary_mask_prof(curr_mask, dist_smooth_angle, apod_start_dist, apod_end_dist)
            curr_mask_smoothed = curr_mask_smoothed / np.max(curr_mask_smoothed)
            hmask_smoothed_dic[cluster_mask_radius_am] = curr_mask_smoothed
            #np.save('tsz_apodised_mask.npy', hmask_smoothed_dic)
        print('\n\t\tdone')
    else:
        hmask_smoothed_dic = copy.deepcopy(hmask_dic)


    print('\n\tcreate final mask')
    final_hmask_arr = list( hmask_smoothed_dic.values() )
    final_hmask = np.product( final_hmask_arr , axis = 0)
    final_hmask = final_hmask / np.max(final_hmask)


    return final_hmask




#apod mask
def apodize_binary_mask_prof(
    binary_mask, dist_smooth_angle, apod_start_dist, apod_end_dist
):
    """
    Apodize a binary mask by applying a smooth profile near the boundaries.


    The profile is [x-sin(x)]/2pi in the range [0, 2pi], which is an integral
    of a cross section of the cosine kernel used in the method above, i.e.,
    apodize_binary_mask_conv. (The cross section itself is [1 + cos(x)]/2 in
    the range [-pi, pi].)


    In this method, first, a distance map is created based on the binary mask.
    In this distance map, the value of a pixel represents the distance between
    the corresponding pixel of the binary mask and the nearest masked pixel
    (value 0.0) of the mask. Then, this distance map is smoothed by a Gaussian
    kernel. Finally, the aforementioned profile is applied to the region of the
    binary mask whose counterpart in the smoothed distance map has distance
    values within a certain range.


    If the binary mask represents a field, then the method
    apodize_binary_mask_conv seems to be better than this one because the
    former seems to create a smoother mask in the sense that there is less
    power in the spectrum of the mask beyond ell of a few thousands.


    On the other hand, if the binary mask represents point sources,
    then this latter method seems to create a better-looking apodization mask
    in the sense that the smooth region around each disk masking a source looks
    more circularly symmetric for a high Nside.


    Parameters:
    -----------
    binary_mask : array
        An array that corresponds to a full-sky HEALPix map representing a
        binary mask.
    dist_smooth_angle : float
        The FWHM of the Gaussian kernel used to smooth the distance map.
        G3Units need to be used when this parameter is specified.
    apod_start_dist : float
        The region of the binary mask whose counterpart in the smoothed
        distance map has distance values smaller than the value specified by
        this parameter will not get the profile applied. G3Units need to be
        used when this parameter is specified.
    apod_end_dist : float
        The region of the binary mask whose counterpart in the smoothed
        distance map has distance values larger than the value specified by
        this parameter will not get the profile applied. G3Units need to be
        used when this parameter is specified.


    Returns:
    --------
    smooth_mask : array:
        An array that corresponds to a full-sky HEALPix map representing the
        apodized mask.
    """
    import healpy as hp


    '''
    dist_smooth_angle /= core.G3Units.rad
    apod_start_dist /= core.G3Units.rad
    apod_end_dist /= core.G3Units.rad
    '''
    net_dist = apod_end_dist - apod_start_dist


    # Construct the distance map and smooth it
    dist_map = hp.dist2holes(binary_mask)
    binary_mask[dist_map <= apod_start_dist] = 0.0


    smooth_region = (dist_map > apod_start_dist) & (dist_map < apod_end_dist)
    #20220510 - too slow for 3*nside-1
    #dist_map = hp.smoothing(dist_map, fwhm=dist_smooth_angle)
    nside = H.get_nside(binary_mask)
    dist_map = hp.smoothing(dist_map, fwhm=dist_smooth_angle, lmax = nside)


    # Apply the profile and take care of a few small things
    smooth_mask = np.array(binary_mask)
    x = (dist_map[smooth_region] - apod_start_dist) / net_dist * 2.0 * np.pi
    del dist_map
    smooth_mask[smooth_region] = (x - np.sin(x)) / (2.0 * np.pi)


    smooth_mask *= binary_mask
    smooth_mask[smooth_mask < 0.0] = 0.0
    smooth_mask[smooth_mask > 1.0] = 1.0


    del binary_mask
    return smooth_mask


import numpy as np, sys, os, healpy as H, glob


########################################
########################################
########################################
#### Cluster mask


if (1): #mdpl2 - M500c vs z mask for SPT 100d megadeep


    m500c_threshold = -1
    expname = 'SPT-3G'
    cluster_lmz_dic_fname = 'cluster_limiting_masses_M500c_vs_z.npy'
    cluster_lmz_dic = pickle.load(open(cluster_lmz_dic_fname, 'rb'))[expname]


    nside = 8192
    hmask = get_apodised_mdpl2_cluster_mask(nside, m500c_threshold = m500c_threshold, cluster_lmz_dic = cluster_lmz_dic, expname = expname)#, cluster_mask_radius_am = cluster_mask_radius_am)
    sys.exit()




if (1): #mdpl2 - fixed mass
    nside = 8192
    hmask = get_apodised_mdpl2_cluster_mask(nside, m500c_threshold = 5e13, cluster_lmz_dic = None) #tsz mask




########################################
########################################
########################################
#### Source mask


def get_mdpl2_conversion_factors_K_to_MjyperSr(expname, band):
    mdpl2_MjyperSr_to_K_conv_planck_dic = {100: 243.623, 143: 371.036, 217: 481.882, 353: 287.281, 545: 57.6963, 857: 2.26476}
    mdpl2_MjyperSr_to_K_conv_spt_dic = {95: 208.973, 150: 375.876, 220: 472.522, 221: 473.332, 285: 414.977, 286: 414.977, 345: 310.827}    
    mdpl2_MjyperSr_to_K_conv_s4_dic = {145: 379.391*0.976, 155: 403.379*0.975}
    if expname is None: #20230218
        #assuming same factors for 285 and 286 GHz bands.
        mdpl2_MjyperSr_to_K_conv_dic = {95: 208.973, 150: 375.876, 220: 472.522, 285: 414.977, 345: 310.827}    
    if expname == 'planck':
        curr_dic = mdpl2_MjyperSr_to_K_conv_planck_dic
    elif expname == 'spt3g' or expname == 'spt' or expname == 'spt4':
        curr_dic = mdpl2_MjyperSr_to_K_conv_spt_dic
    elif expname == 'cmbs4' or expname == 's4wide' or expname == 's4deep':
        curr_dic = mdpl2_MjyperSr_to_K_conv_s4_dic
    elif expname is None:
        curr_dic = mdpl2_MjyperSr_to_K_conv_dic


    return curr_dic[band]


#this is the logic
hmap_cib_in_uK = H.read_map( hmap_cib_fname ) #get CIB map at 150 GHz
hmap_rad_in_uK = H.read_map( hmap_rad_fname ) #get Radio map at 150 GHz
hmap_sources_in_uK = hmap_cib_in_uK + hmap_rad_in_uK #combine them


#convert into flux units
band0 = 150
threshold_mjy_band0 = 6. #mJy
conv_factor_K_to_mjyperSr = get_mdpl2_conversion_factors_K_to_MjyperSr(expname, band0) #conversion from K to mJy/Sr
#print(conv_factor_K_to_mjyperSr)
uK_to_K = 1./1e6
hmap_sources_in_mjy_per_sr = np.copy(curr_hmap) * uK_to_K * conv_factor_K_to_mjyperSr


masked_pixels = get_point_source_mask_in_healpix(band0, hmap_sources_in_mjy_per_sr, threshold_mjy_band0)
hmask = np.ones( len(hmap_sources_in_uK) )
hmask[masked_pixels] = 0.

    hmask = get_apodised_mdpl2_cluster_mask(nside, m500c_threshold = m500c_threshold, cluster_lmz_dic = cluster_lmz_dic, expname = expname)#, cluster_mask_radius_am = cluster_mask_radius_am)
    sys.exit()




if (1): #mdpl2 - fixed mass
    nside = 8192
    hmask = get_apodised_mdpl2_cluster_mask(nside, m500c_threshold = 5e13, cluster_lmz_dic = None) #tsz mask




########################################
########################################
########################################
#### Source mask


def get_mdpl2_conversion_factors_K_to_MjyperSr(expname, band):
    mdpl2_MjyperSr_to_K_conv_planck_dic = {100: 243.623, 143: 371.036, 217: 481.882, 353: 287.281, 545: 57.6963, 857: 2.26476}
    mdpl2_MjyperSr_to_K_conv_spt_dic = {95: 208.973, 150: 375.876, 220: 472.522, 221: 473.332, 285: 414.977, 286: 414.977, 345: 310.827}    
    mdpl2_MjyperSr_to_K_conv_s4_dic = {145: 379.391*0.976, 155: 403.379*0.975}
    if expname is None: #20230218
        #assuming same factors for 285 and 286 GHz bands.
        mdpl2_MjyperSr_to_K_conv_dic = {95: 208.973, 150: 375.876, 220: 472.522, 285: 414.977, 345: 310.827}    
    if expname == 'planck':
        curr_dic = mdpl2_MjyperSr_to_K_conv_planck_dic
    elif expname == 'spt3g' or expname == 'spt' or expname == 'spt4':
        curr_dic = mdpl2_MjyperSr_to_K_conv_spt_dic
    elif expname == 'cmbs4' or expname == 's4wide' or expname == 's4deep':
        curr_dic = mdpl2_MjyperSr_to_K_conv_s4_dic
    elif expname is None:
        curr_dic = mdpl2_MjyperSr_to_K_conv_dic


    return curr_dic[band]


#this is the logic
hmap_cib_in_uK = H.read_map( hmap_cib_fname ) #get CIB map at 150 GHz
hmap_rad_in_uK = H.read_map( hmap_rad_fname ) #get Radio map at 150 GHz
hmap_sources_in_uK = hmap_cib_in_uK + hmap_rad_in_uK #combine them


#convert into flux units
band0 = 150
threshold_mjy_band0 = 6. #mJy
conv_factor_K_to_mjyperSr = get_mdpl2_conversion_factors_K_to_MjyperSr(expname, band0) #conversion from K to mJy/Sr
#print(conv_factor_K_to_mjyperSr)
uK_to_K = 1./1e6
hmap_sources_in_mjy_per_sr = np.copy(curr_hmap) * uK_to_K * conv_factor_K_to_mjyperSr


masked_pixels = get_point_source_mask_in_healpix(band0, hmap_sources_in_mjy_per_sr, threshold_mjy_band0)
hmask = np.ones( len(hmap_sources_in_uK) )
hmask[masked_pixels] = 0.


## Cluster masking

The tSZ map is to have clusters with mass $M_{500}\ge3\times10^{14}$ are to be masked with radii ranging from $3\theta_{500}$ to $5\theta_{500}$ with a minimum radius of 10 arcminutes.

## Masking with sigma clipping

Until we get access to cluster and point source masks, we use sigma clipping. The process is to compute $\mu$ and $\sigma$, remove all pixels above $\mu+10\times\sigma$ and iterate until no more pixels are removed by this process. Input maps have an NSIDE of 8192, corresponding to 805306368 pixels that are 0.43 arcminutes.

In [ ]:
import numpy as np
import astropy.units as u
import healpy as hp
import os
import matplotlib.pyplot as plt

In [ ]:
data_dir = 'data'
plots_dir = 'plots'
os.makedirs(plots_dir, exist_ok=True)

cib_map = hp.read_map(os.path.join(data_dir, 'mdpl2_len_mag_cibmap_act_150_uk.fits'), memmap=True)

hp.visufunc.mollview(cib_map, title='CIB Map at 150 GHz', unit='K_CMB')
plt.savefig(os.path.join(plots_dir, 'cib_map_150ghz.png'), dpi=300)

tsz_map = hp.read_map(os.path.join(data_dir, 'mdpl2_ltszNG_bahamas80_rot_sum_4_176_bnd_unb_1.0e+12_1.0e+18_v103021_lmax24000_nside8192_interp1.0_method1_1_lensed_map.fits'), memmap=True)

hp.visufunc.mollview(tsz_map, title='tSZ Map at 150 GHz', unit='K_CMB')
plt.savefig(os.path.join(plots_dir, 'tsz_map_150ghz.png'), dpi=300)

# Get pixel info:
nside = hp.get_nside(cib_map)
npix = hp.nside2npix(nside)
pix_size = hp.nside2resol(nside, arcmin=True)
print(f"NSIDE: {nside}, NPIX: {npix}, Pixel Size: {pix_size:.2f} arcmin")

# Plot histogram of pixel values:
harsh_clip = 1e-5
cib_map_clipped = np.clip(cib_map, -harsh_clip, harsh_clip)
plt.figure(figsize=(10, 6))
plt.hist(cib_map_clipped.flatten(), bins=100, alpha=0.5, label='CIB Map')
plt.hist(tsz_map.flatten(), bins=100, alpha=0.5, label='tSZ Map')
plt.xlabel('Temperature (K_CMB)')
plt.ylabel('Number of Pixels')
plt.yscale('log')
plt.title('Pixel Value Distribution')
plt.legend()
plt.savefig(os.path.join(plots_dir, 'pixel_value_histogram.png'), dpi=300)

In [ ]:
# Get CIB mask using sigma clipping
from astropy.stats import sigma_clip
cib_mask = sigma_clip(cib_map, sigma=10, maxiters=5).mask
hp.fitsfunc.write_map(os.path.join(data_dir, 'cib_ptsrc_mask.fits'), cib_mask.astype(float), overwrite=True)
hp.visufunc.mollview(cib_mask.astype(float), title='CIB Mask (Sigma Clipped)', unit='Mask')
plt.savefig(os.path.join(plots_dir, 'cib_ptsrc_mask.png'), dpi=300)

cib_150_masked = np.where(cib_mask, 0, cib_map)
hp.fitsfunc.write_map(os.path.join(data_dir, 'cib_150_ptsrc_masked.fits'), cib_150_masked, overwrite=True)
hp.visufunc.mollview(cib_150_masked, title='CIB Map with Mask Applied', unit='K_CMB')
plt.savefig(os.path.join(plots_dir, 'cib_map_ptsrc_masked.png'), dpi=300)

# Get tSZ mask using sigma clipping
tsz_mask = sigma_clip(tsz_map, sigma=10, maxiters=5).mask
hp.fitsfunc.write_map(os.path.join(data_dir, 'tsz_ptsrc_mask.fits'), tsz_mask.astype(float), overwrite=True)
hp.visufunc.mollview(tsz_mask.astype(float), title='tSZ Mask (Sigma Clipped)', unit='Mask')
plt.savefig(os.path.join(plots_dir, 'tsz_ptsrc_mask.png'), dpi=300)

tsz_150_masked = np.where(tsz_mask, 0, tsz_map)
hp.fitsfunc.write_map(os.path.join(data_dir, 'tsz_150_ptsrc_masked.fits'), tsz_150_masked, overwrite=True)
hp.visufunc.mollview(tsz_150_masked, title='tSZ Map with Mask Applied', unit='K_CMB')
plt.savefig(os.path.join(plots_dir, 'tsz_map_ptsrc_masked.png'), dpi=300)


## Low pass filter

Both maps have a low pass filter applied, with $\ell\ge7000$ set to zero.

In [ ]:
# Low pass both maps with ell > 7000 set to zero
lmax = 7000
cib_150_lowpassed = hp.sphtfunc.alm2map(hp.sphtfunc.map2alm(cib_150_masked, lmax=lmax), nside=nside)
tsz_150_lowpassed = hp.sphtfunc.alm2map(hp.sphtfunc.map2alm(tsz_150_masked, lmax=lmax), nside=nside)
hp.fitsfunc.write_map(os.path.join(data_dir, 'cib_150_ptsrc_masked_lowpassed.fits'), cib_150_lowpassed, overwrite=True)
hp.fitsfunc.write_map(os.path.join(data_dir, 'tsz_150_ptsrc_masked_lowpassed.fits'), tsz_150_lowpassed, overwrite=True)
hp.visufunc.mollview(cib_150_lowpassed, title='CIB Map Lowpassed (l > 7000)', unit='K_CMB')
plt.savefig(os.path.join(plots_dir, 'cib_map_lowpassed.png'), dpi=300)
hp.visufunc.mollview(tsz_150_lowpassed, title='tSZ Map Lowpassed (l > 7000)', unit='K_CMB')
plt.savefig(os.path.join(plots_dir, 'tsz_map_lowpassed.png'), dpi=300)

## Creating map patches

In [ ]:
import astropy.units as u

RES = 256
STEP_SIZE = 6 * u.deg
ANG_X = 6. * u.deg
ANG_Y = 6. * u.deg

@u.quantity_input
def get_patch_centers(gal_cut: u.deg, step_size: u.deg):
    """ Function to get the centers of the various patches to be cut out.

    Parameters
    ----------
    gal_cut: float
        We will miss out the region +/- `gal_cut` in Galactic latitude, measured
        in degrees.
    step_size: float
        Stepping distance in Galactic longitude, measured in degrees, between 
        patches.

    Returns
    -------
    list(tuple(float))
        List of two-element tuples containing the longitude and latitude.
    """
    gal_cut = gal_cut.to(u.deg)
    step_size = step_size.to(u.deg)
    assert gal_cut.unit == u.deg
    assert step_size.unit == u.deg
    southern_lat_range = np.arange(-90, (-gal_cut-step_size).value, step_size.value) * u.deg
    northern_lat_range = np.arange((gal_cut + step_size).value, 90, step_size.value) * u.deg
    lat_range = np.concatenate((southern_lat_range, northern_lat_range))

    centers = []
    for t in lat_range:
        step = step_size.value / np.cos(t.to(u.rad).value)
        for i in np.arange(0, 360, step):
            centers.append((i * u.deg, t))
    return centers

class FlatCutter(object):
    """ Object to control the extraction of flat patches from a given HEALPix 
    map.

    Object is initialized with parameters defining the geometry of the patch:
    its length in degrees, and the number of pixels in each direction. 
    The `rotate_and_interpolate` method defines a grid centered at (0, 0)
    of dimensions corresponding to `xlen`, `ylen`, and rotates it to the 
    point (lon, lat). The value of the map at the resulting grid of longitudes 
    and latitudes is then determined by interpolation. 
    """
    @u.quantity_input
    def __init__(self, ang_x: u.deg, ang_y: u.deg, xres, yres):
        assert type(xres) is int
        assert type(yres) is int
        self.xres = xres
        self.yres = yres

        self.ang_x = ang_x
        self.ang_y = ang_y
        
        # get grid of unit vectors corresponding to flat patch around
        # pole (z = 1). For this we use ang_x in radians, as is appropriate
        # for the implicit small-angle approximation 
        self.xarr = np.linspace(- self.ang_x.to(u.rad).value / 2., 
                                self.ang_x.to(u.rad).value / 2., xres)
        self.yarr = np.linspace(- self.ang_y.to(u.rad).value / 2., 
                                self.ang_y.to(u.rad).value / 2., yres)

        xgrid, ygrid = np.meshgrid(self.xarr, self.yarr)
        xgrid = xgrid.ravel()[None, :]
        ygrid = ygrid.ravel()[None, :]
        zgrid = np.ones_like(ygrid)
        
        # vectors corresponding to cartesian grid around poll
        self.vecs = np.concatenate((xgrid, ygrid, zgrid)).T

        # get the latitude (*not colatitude*) and longitude in degrees
        # of the cartesian grid points around the pole. 
        self.lons, self.lats = hp.vec2ang(self.vecs, lonlat=True)
        self.lats *= u.deg
        self.lons *= u.deg
        return
    
    @u.quantity_input
    def rotate_to_pole_and_interpolate(self, lon: u.deg, lat: u.deg, ma):
        """ Method to rotate the grid at (0, 0) to `rot=(lon, lat)`, and sample
        the map at the grid points by interpolation.

        Parameters
        ----------
        lat, lon: float
            Latitude (*not* colatitude) and longitude of point to be rotated
            to the North pole, in degrees.
        ma: ndarray
            Healpix map from which the interpolation is to be made.
        """
        if hp.pixelfunc.maptype(ma) == 0:  # a single map is converted to a list
            ma = [ma]
        # define a rotation object in terms of the theta_rot and phi_rot angles.
        # This returns a rotator object that can be applied to rotate a given
        # vector by this angle. Since we are interested in rotating some patch
        # to the pole, we actually want to apply the *inverse* rotation operator
        # to the vectors self.co_lats, self.lons.
        lon = lon.to(u.deg)
        lat = lat.to(u.deg)
        rotator = hp.Rotator(rot=[lon.value, lat.value - 90.], deg=True)
        self.inv_lon_grid, self.inv_lat_grid = rotator.I(self.lons.to(u.deg).value, self.lats.to(u.deg).value, lonlat=True)
        # Interpolate the original map to the pixels centers in the new ref frame
        m_rot = [hp.get_interp_val(each, self.inv_lon_grid, self.inv_lat_grid, lonlat=True) for each in ma]

        # Rotate polarization
        if len(m_rot) > 1:
            # Create a complex map from QU  and apply the rotation in psi due to the rotation
            # Slice from the end of the array so that it works both for QU and IQU
            m_rot[-2], m_rot[-1] = spin2rot(m_rot[-2], m_rot[-1], rotator.angle_ref(self.inv_lon_grid, self.inv_lat_grid, lonlat=True))
            m_rot[-2], m_rot[-1] = spin2rot(m_rot[-2], m_rot[-1], self.lons.to(u.rad).value)
        else:
            m_rot = m_rot[0]
        return np.moveaxis(np.array(m_rot).reshape(-1, self.xres, self.yres), 0, -1)

In [ ]:
RES = 256
STEP_SIZE = 6 * u.deg
ANG_X = 6. * u.deg
ANG_Y = 6. * u.deg
ptsrc = 2
flatskymapparams = [RES, RES, 60.*ANG_X.value/RES, 60.*ANG_Y.value/RES] #Code requires pixelres to be in arcmin
flatskymapparams

In [ ]:
final_cib_150_map = hp.fitsfunc.read_map(os.path.join(data_dir, 'cib_150_ptsrc_masked_lowpassed.fits'))
centers = get_patch_centers(gal_cut=0.*u.deg, step_size=STEP_SIZE)
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cib_cut_maps = np.array([cutter.rotate_to_pole_and_interpolate(lon, lat, final_cib_150_map) for (lon, lat) in centers], dtype=np.float32)

fname = f"data/low_pass/{ptsrc}mJy/cut_maps_RES_{RES}_ANG_X_{ANG_X.value}_deg_{ptsrc}mJy_lp.npy"
np.save(fname,cib_cut_maps)

def apply_maxmin_normalization(maps):
    min_val = np.min(maps)
    max_val = np.max(maps)
    return (maps - min_val) / (max_val - min_val) 

processed_maps = np.copy(cut_maps)

min_val = np.min(cut_maps)
max_val = np.max(cut_maps)
print (min_val,max_val - min_val)

processed_maps = apply_maxmin_normalization(processed_maps)

final_tsz_map = hp.fitsfunc.read_map(os.path.join(data_dir, 'tsz_150_ptsrc_masked_lowpassed.fits'))

In [ ]:
processed_maps = apply_maxmin_normalization(processed_maps)
fpath = "data/low_pass/{:d}mJy/CIB_map_150GHz_{:d}_st{:d}_minmax_{:d}mJy_zero_lp.npy".format(ptsrc,RES, int(STEP_SIZE.value),ptsrc)
#fpath = "data/low_pass/{:d}mJy/tSZ3_map_150GHz_{:d}_st{:d}_minmax_{:d}mJy_norm_lp.npy".format(ptsrc,RES, int(STEP_SIZE.value),ptsrc)
print(fpath)
np.save(fpath, processed_maps)

## Gaussian patches

Create patches from the same power spectrum as the AGORA CIB map.

In [ ]:
cib_map_meansub = final_cib_150_map - np.mean(cib_150_map)
cl_from_map = hp.anafast(cib_map_meansub, lmax=13000)
gaussian_map = hp.synfast(cl_from_map, 8192)
centers = get_patch_centers(gal_cut=0*u.deg, step_size=STEP_SIZE)
print("Total patches: ", len(centers))
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cut_maps = np.array([cutter.rotate_to_pole_and_interpolator(lon, lat, gaussian_map) for (lon, lat) in centers], dtype=np.float32)
np.save("data/low_pass/2mJy/cut_maps_RES_256_ANG_X_6.0 deg_2mJy_lp_gaussian_tsz.npy", cut_maps)

## Joint Gaussian maps

Record auto and cross power spectra and patches from original AGORA CIB and tSZ maps.

In [ ]:
cib_map_meansub = cib_150_map - np.mean(cib_150_map)
cib_cl_from_map = hp.anafast(cib_map_meansub, lmax=13000)
tsz_map_meansub = tsz_150_map - np.mean(tsz_150_map)
tsz_cl_from_map = hp.anafast(tsz_map_meansub, lmax=13000)
cib_tsz_cl_from_map = hp.anafast(cib_map_meansub,tsz_map_meansub, lmax=13000)
hp.write_cl("data/low_pass/2mJy/anafast_tsz3_cl_from_map.fits",tsz_cl_from_map)
hp.write_cl("data/low_pass/2mJy/anafast_cib_cl_from_map.fits",cib_cl_from_map)
hp.write_cl("data/low_pass/2mJy/anafast_cib_tsz3_cl_from_map.fits",cib_tsz_cl_from_map)
cib_cl_from_map=hp.read_cl("data/low_pass/2mJy/anafast_cib_cl_from_map.fits")
tsz_cl_from_map=hp.read_cl("data/low_pass/2mJy/anafast_tsz3_cl_from_map.fits")
cib_tsz_cl_from_map=hp.read_cl("data/low_pass/2mJy/anafast_cib_tsz3_cl_from_map.fits")
# Ordering for healpy.synfast:
# [C_ℓ^TT (aa), C_ℓ^EE (bb), C_ℓ^BB (ignored), C_ℓ^TE (ab)]
cls = [cib_cl_from_map, tsz_cl_from_map, np.zeros_like(cib_cl_from_map), cib_tsz_cl_from_map]
maps = hp.synfast(cls, nside=8192, new=True, pol=False)
cib_map = maps[0]
tsz_map = maps[1]
centers = get_patch_centers(gal_cut=0.*u.deg, step_size=STEP_SIZE)
print("Total patches: ",len(centers))
# cut out maps at each of the patch centers
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cut_maps = np.array([cutter.rotate_to_pole_and_interpolate(lon, lat, cib_map) for (lon, lat) in centers], dtype=np.float32)
np.save("data/low_pass/2mJy/cut_maps_RES_256_ANG_X_6.0 deg_2mJy_lp_gaussian_cib_joint3.npy",cut_maps)
centers = get_patch_centers(gal_cut=0.*u.deg, step_size=STEP_SIZE)
print("Total patches: ",len(centers))
# cut out maps at each of the patch centers
cutter = FlatCutter(ANG_X, ANG_Y, RES, RES)
cut_maps = np.array([cutter.rotate_to_pole_and_interpolate(lon, lat, tsz_map) for (lon, lat) in centers], dtype=np.float32)
np.save("data/low_pass/2mJy/cut_maps_RES_256_ANG_X_6.0 deg_2mJy_lp_gaussian_tsz_joint3.npy",cut_maps)